<a href="https://colab.research.google.com/github/Harshu0810/Q-A-with-files/blob/main/rag_bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q \
  llama-index-core \
  llama-index-embeddings-huggingface \
  llama-index-llms-huggingface \
  sentence-transformers \
  transformers \
  accelerate \
  bitsandbytes \
  gradio \
  pypdf \
  gdown \
  tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.5/331.5 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 18.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>

In [2]:
INDEX_CACHE = {}

In [3]:
import os
import re
import hashlib
from tqdm import tqdm

from llama_index.core import (
    SimpleDirectoryReader,
    VectorStoreIndex,
)
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface import HuggingFaceLLM

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [4]:
def extract_drive_id(url: str) -> str:
    patterns = [
        r"/file/d/([a-zA-Z0-9_-]+)",
        r"/folders/([a-zA-Z0-9_-]+)"
    ]
    for p in patterns:
        m = re.search(p, url)
        if m:
            return m.group(1)
    raise ValueError("Invalid Google Drive link")

In [5]:
def download_from_drive(link: str, target_dir="data"):
    os.makedirs(target_dir, exist_ok=True)
    file_id = extract_drive_id(link)

    # Folder download
    if "folders" in link:
        !gdown --folder {link} -O {target_dir}
        return target_dir

    # Single file
    out_path = os.path.join(target_dir, "input.pdf")
    !gdown --id {file_id} -O {out_path}
    return out_path

In [6]:
embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
    load_in_4bit=True
)

llm = HuggingFaceLLM(
    model=llm_model,
    tokenizer=tokenizer,
    context_window=4096,
    max_new_tokens=256,
    generate_kwargs={"temperature": 0.2},
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [8]:
def build_query_engine(drive_link: str):
    link_hash = hashlib.md5(drive_link.encode()).hexdigest()

    if link_hash in INDEX_CACHE:
        return INDEX_CACHE[link_hash], "✅ Using cached index"

    path = download_from_drive(drive_link)

    reader_kwargs = (
        {"input_files": [path]} if os.path.isfile(path)
        else {"input_dir": path, "recursive": True}
    )

    documents = SimpleDirectoryReader(**reader_kwargs).load_data()

    index = VectorStoreIndex.from_documents(
        documents,
        embed_model=embed_model
    )

    query_engine = index.as_query_engine(
        llm=llm,
        response_mode="compact",
        similarity_top_k=2
    )

    INDEX_CACHE[link_hash] = query_engine
    return query_engine, "✅ Documents indexed successfully"

In [9]:
import gradio as gr

query_engine = None

def load_docs(link):
    global query_engine
    try:
        query_engine, msg = build_query_engine(link)
        return msg
    except Exception as e:
        return f"❌ Error: {str(e)}"

def chat_fn(question, history):
    if query_engine is None:
        history.append(("System", "Please load a Drive link first."))
        return history

    response = query_engine.query(question)
    history.append((question, str(response)))
    return history

with gr.Blocks() as demo:
    gr.Markdown("# 📄 Universal Drive RAG Chatbot")
    gr.Markdown(
        "Paste **any public Google Drive file or folder link**.\n\n"
    )

    drive_link = gr.Textbox(label="Google Drive File or Folder Link")
    load_btn = gr.Button("Load & Index Documents")
    status = gr.Textbox(label="Status", interactive=False)

    chatbot = gr.Chatbot(height=420)
    question = gr.Textbox(label="Ask a question")
    clear = gr.Button("Clear Chat")

    load_btn.click(load_docs, drive_link, status)
    question.submit(chat_fn, [question, chatbot], chatbot)
    clear.click(lambda: [], None, chatbot)

demo.launch(share=True)

/tmp/ipython-input-1534/3251110818.py:32: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=420)
/tmp/ipython-input-1534/3251110818.py:32: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=420)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7897bb981df51b3151.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
